In [1]:
import duckdb
import os
from pathlib import Path

os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())

con = duckdb.connect(config={'memory_limit': '2GB'})

print("=== Simple Star Schema Test ===")

# 1. Create Staging with is_high_risk
print("Creating stg_payments...")
con.sql("""
    CREATE OR REPLACE TABLE stg_payments AS 
    SELECT 
        transaction_id,
        timestamp,
        user_id,
        merchant_id,
        amount,
        currency,
        status,
        payment_method,
        risk_score,
        DATE(timestamp) as transaction_date,
        CASE WHEN risk_score > 0 THEN TRUE ELSE FALSE END as is_high_risk
    FROM 'data/silver/date=*/payments_silver.parquet'
    LIMIT 50000
""")

# 2. Create simple Fact (no complex joins first)
print("Creating fact_payments (simplified)...")
con.sql("""
    CREATE OR REPLACE TABLE fact_payments AS 
    SELECT 
        ROW_NUMBER() OVER () as payment_key,
        transaction_id,
        amount,
        risk_score,
        status,
        is_high_risk,
        DATE(timestamp) as transaction_date
    FROM stg_payments
""")

print("Success!")
con.sql("SELECT COUNT(*) as row_count, AVG(risk_score) as avg_risk FROM fact_payments").show()

Working directory: C:\Users\ThinkPad\Documents\de\payments-dimensional-modeling
=== Simple Star Schema Test ===
Creating stg_payments...
Creating fact_payments (simplified)...
Success!
┌───────────┬──────────┐
│ row_count │ avg_risk │
│   int64   │  double  │
├───────────┼──────────┤
│     50000 │    3.542 │
└───────────┴──────────┘

